# Real vs simulated vs generated events

One recorded human action as three kinds of event data: real DAVIS346 events, events simulated from a 240 fps phone video (ESIM-style, notebook 06) and event images generated by diffusion from the person's 2D pose (ControlEvents, notebook 09). No training, no new methods.

Set the action in section 0 and run all: the two clips are aligned in time automatically, the person is found by OpenPose in both cameras, and the checklist under section 0 says what was found and what will run. Generation needs a GPU: use a Colab GPU runtime (on Apple silicon each image takes about 70 s). Generated images and keypoints are cached next to the data, so later runs only redo the comparison. Sections whose data is missing print `SKIPPED: <path> not found`.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    REPO = "/content/event-perception"
    if not os.path.exists(REPO):
        !git clone https://github.com/allusson/event-perception.git {REPO}
    %cd {REPO}
    !pip install -q -e ".[diffusion]"
    sys.path.insert(0, REPO)
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/event-perception-data/human"
else:
    # local: assumes `pip install -e ".[diffusion]"` was run from the repo root
    os.chdir(os.path.dirname(os.path.abspath("")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
    DATA_DIR = "data/human"

In [ ]:
import dataclasses, hashlib, time, warnings
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display
from PIL import Image
from diffusers.utils import logging as diffusers_logging
diffusers_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

from evcam import controlevents as ce
from evcam.align import (activity_box, check_sync, describe, grow_box, map_by_features, map_by_keypoints, motion_start,
                         plot_alignment, resolve_sync, sync_by_event_rate)
from evcam.sources import FFMPEG_HINT, load_davis, load_video
from evcam.simulate import simulate_events

DEVICE, DTYPE = ce.pick_device()
SIZE = ce.SIZE             # generation canvas, px
VIDEO_LONG_SIDE = 480      # phone frames are loaded at this long side (px) for the alignment and the simulation
SIM_MARGIN_S = 0.2         # simulated before and after the analysed span, s
CROP_MARGIN = 0.15         # person box grown by this fraction of its size on each side
print(f"device {DEVICE}, data in {DATA_DIR}")


def disp(img):
    # display only: linear intensity -> gamma 1/2.2
    return np.clip(img, 0, 1) ** (1 / 2.2)


def in_window(ev, t0_us, t1_us):
    return ev[(ev["t"] >= t0_us) & (ev["t"] < t1_us)]


def count_image(ev, wh):
    """Events per pixel, polarity ignored."""
    img = np.zeros((wh[1], wh[0]), np.float32)
    np.add.at(img, (ev["y"], ev["x"]), 1)
    return img


def white_on_black(img):
    """Scale to [0, 1] by the 99th percentile of the non-zero pixels: white events on black."""
    v = np.percentile(img[img > 0], 99) if (img > 0).any() else 1.0
    return np.clip(img / v, 0, 1)


def read_frames(path, indices):
    """Full-resolution colour frames (RGB uint8) at the given frame indices, read in order."""
    want, out, k = set(int(i) for i in indices), {}, 0
    cap = cv2.VideoCapture(path)
    ok, bgr = cap.read()
    while ok and k <= max(want):
        if k in want:
            out[k] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        ok, bgr = cap.read()
        k += 1
    cap.release()
    return out


detector, poses = None, {}


def joints_px(key, rgb, scale=1.0):
    """OpenPose 13 joints in the pixels of `rgb` (NaN = missing joint), None if no person; OpenPose runs on
    the image resized by `scale`. Cached in `poses` under `key`."""
    global detector
    if REGENERATE or key not in poses:
        if detector is None:
            detector = ce.openpose_detector()
        h, w = rgb.shape[:2]
        img = cv2.resize(rgb, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
        xy = ce.detect_joints(detector, img)
        poses[key] = np.full((13, 2), np.nan) if xy is None else xy * [w, h]
    return None if np.isnan(poses[key]).all() else poses[key]


def person_box(joints, wh):
    """Union of the joint boxes of all frames, grown by CROP_MARGIN, or None without any person."""
    xy = np.concatenate([j for j in joints if j is not None]) if any(j is not None for j in joints) else None
    if xy is None:
        return None
    x0, y0 = np.nanmin(xy, 0)
    x1, y1 = np.nanmax(xy, 0)
    return grow_box((x0, y0, x1, y1), CROP_MARGIN, wh)


def on_canvas(xy, box):
    """Joints in image pixels -> [0, 1] coordinates of the canvas `box`."""
    return None if xy is None else (xy - np.array(box[:2], float)) / (box[2] - box[0])


T_NOTEBOOK = time.time()

## 0. Configuration

The only cell to edit. Overrides left as `None` are found automatically.

In [ ]:
ACTION = "wave"            # reads <DATA_DIR>/<ACTION>.mov (or .mp4) and <DATA_DIR>/<ACTION>.aedat4

# Overrides: leave as None for automatic
SYNC = None    # or (t_phone_s, t_davis_s): one moment that is the same in both clips, read off the alignment plot in section 2

# Advanced (usually leave as is)
CAPTURE_FPS = 240          # true capture rate of the phone clip
GEN_FPS = 10               # rate of the compared frames, Hz
ACC_MS = 15                # real and simulated events accumulated over this window, centred on each frame time
CLIP_S = 3.0               # analysed span after the motion start, s
C_SIM = 0.2                # contrast threshold of the simulation
NUM_STEPS = 30             # UniPC steps
GUIDANCE = 7.5             # classifier-free guidance scale
SEED = 0
REGENERATE = False         # True: recompute keypoints and images even if cached

### Checklist

Loads both clips and finds everything automatic: the time offset (section 2), the motion start, the analysis window of `CLIP_S` seconds from there, the crop around the person in each camera (OpenPose bounding box over the analysed frames plus a margin; for the DAVIS, OpenPose on the APS frames, or the box of the events if it finds no person) and the spatial mapping between the cameras used for one overlay in section 6. Keypoints are cached in `<DATA_DIR>/cache/<ACTION>/pose.npz`.

In [ ]:
t_sec = time.time()
check_sync(SYNC)
for k, v in {"GEN_FPS": GEN_FPS, "ACC_MS": ACC_MS, "CLIP_S": CLIP_S, "C_SIM": C_SIM, "CAPTURE_FPS": CAPTURE_FPS}.items():
    if not (isinstance(v, (int, float)) and v > 0):
        raise ValueError(f"{k} must be a number > 0; got {v!r}")

VIDEO_PATH = next((f"{DATA_DIR}/{ACTION}{e}" for e in (".mov", ".MOV", ".mp4", ".MP4")
                   if os.path.exists(f"{DATA_DIR}/{ACTION}{e}")), None)
DAVIS_PATH = f"{DATA_DIR}/{ACTION}.aedat4"
CACHE = f"{DATA_DIR}/cache/{ACTION}"
missing = ([] if VIDEO_PATH else [f"{DATA_DIR}/{ACTION}.mov"]) + ([] if os.path.exists(DAVIS_PATH) else [DAVIS_PATH])
skip = f"{' and '.join(missing)} not found" if missing else None

if not skip:
    cap = cv2.VideoCapture(VIDEO_PATH)
    ok, first = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"OpenCV cannot decode {VIDEO_PATH}. Convert it first: {FFMPEG_HINT}")
    PHONE_WH = (first.shape[1], first.shape[0])                        # full resolution, after rotation
    s = VIDEO_LONG_SIDE / max(PHONE_WH)
    video = load_video(VIDEO_PATH, capture_fps=CAPTURE_FPS, resize_to=(round(PHONE_WH[1] * s), round(PHONE_WH[0] * s)))
    davis = load_davis(DAVIS_PATH)
    H, W = video.shape
    Hd, Wd = davis.shape

    # time: offset, motion start, analysis window
    sync = resolve_sync(sync_by_event_rate(davis.events, video, c=C_SIM), SYNC)
    t_davis0_s = motion_start(sync.curves["t_davis"], sync.curves["rate_davis"], max(sync.offset_s, 0.0))
    start_src = "auto"
    if t_davis0_s is None:
        t_davis0_s, start_src = max(sync.offset_s, 0.0), "auto (no clear rise: start of the phone clip)"
    t_video0_s = t_davis0_s - sync.offset_s
    t_video0, t_davis0 = int(round(t_video0_s * 1e6)), int(round(t_davis0_s * 1e6))
    real_ev = davis.events.copy(); real_ev["t"] -= t_davis0                         # time since motion start, us
    frame_us = np.round(np.arange(int(round(CLIP_S * GEN_FPS))) * 1e6 / GEN_FPS).astype(np.int64)
    frame_idx = np.round((t_video0_s + frame_us / 1e6) * CAPTURE_FPS).astype(int)
    n = int(np.sum((frame_idx >= 0) & (frame_idx < video.n_frames) & (frame_us + ACC_MS * 500 < real_ev["t"][-1])))
    if n == 0:
        raise ValueError(f"SYNC = {SYNC} leaves no overlap between the clips after the motion start ({t_davis0_s:.2f} s on the "
                         "DAVIS clock). SYNC must be None or (t_phone_s, t_davis_s) in seconds, e.g. (1.40, 12.35).")
    frame_us, frame_idx = frame_us[:n], frame_idx[:n]

    # space: OpenPose on the phone frames (full resolution) and on the nearest APS frames
    os.makedirs(CACHE, exist_ok=True)
    POSE_FILE = f"{CACHE}/pose.npz"
    poses = {} if REGENERATE or not os.path.exists(POSE_FILE) else dict(np.load(POSE_FILE))
    phone_full = read_frames(VIDEO_PATH, frame_idx)
    phone_px = [joints_px(f"phonefull_{i}", phone_full[i], scale=0.5) for i in frame_idx]
    crop = person_box(phone_px, PHONE_WH)
    box_phone = ce.canvas_box(PHONE_WH, crop)                                     # full-resolution phone px
    phone_src = (f"auto (OpenPose, person in {sum(j is not None for j in phone_px)} of {n} frames)" if crop
                 else "auto (OpenPose finds no person: centred square)")
    aps_idx = [None] * n
    aps_px = [None] * n
    if davis.frames is not None:
        aps_idx = [int(np.argmin(np.abs(davis.frame_t - (t + t_davis0)))) for t in frame_us]
        to_rgb = lambda g: np.repeat((np.clip(g, 0, 1) * 255).astype(np.uint8)[..., None], 3, axis=2)
        aps_px = [joints_px(f"apsfull_{j}", to_rgb(davis.frames[j]), scale=2.0) for j in aps_idx]
    np.savez(POSE_FILE, **poses)
    crop_d = person_box(aps_px, (Wd, Hd))
    davis_src = f"auto (OpenPose on APS frames, person in {sum(j is not None for j in aps_px)} of {n})"
    if crop_d is None:
        crop_d = activity_box(count_image(in_window(real_ev, 0, frame_us[-1] + ACC_MS * 500), (Wd, Hd)), margin=CROP_MARGIN)
        davis_src = ("auto (box of the events in the window; "
                     + ("no APS frames)" if davis.frames is None else "OpenPose finds no person on the APS frames)"))
    box_davis = ce.canvas_box((Wd, Hd), crop_d)
    box_work = tuple(int(round(v * W / PHONE_WH[0])) for v in box_phone)          # same box at VIDEO_LONG_SIDE
    joints = [on_canvas(j, box_phone) for j in phone_px]                         # canvas coordinates, for the skeletons
    real_joints = [on_canvas(j, box_davis) for j in aps_px]

    # spatial mapping phone px -> DAVIS px (one overlay in section 6): keypoints, else features
    both = [k for k in range(n) if phone_px[k] is not None and aps_px[k] is not None]
    mapping, map_src = None, "none: the overlay in section 6 is skipped"
    m = map_by_keypoints(np.array([phone_px[k] for k in both]), np.array([aps_px[k] for k in both])) if both else None
    if m is not None:
        mapping, map_src = np.vstack([m[0], [0, 0, 1]]), f"keypoints ({len(both)} frames), RMS residual {m[1]:.1f} DAVIS px"
    elif davis.frames is not None:
        best = None
        for k in np.linspace(0, n - 1, min(3, n)).round().astype(int):
            r = map_by_features(phone_full[frame_idx[k]], davis.frames[aps_idx[k]])
            if r is not None and (best is None or r[1] > best[1]):
                best = r
        if best is not None:
            mapping, map_src = best[0], f"features ({best[1]} inliers; keypoints failed)"

    # generated images: cached per crop, window and sampler settings
    TAG = hashlib.md5(repr((box_phone, int(frame_idx[0]), n, GEN_FPS, NUM_STEPS, GUIDANCE)).encode()).hexdigest()[:8]
    GEN_DIR = f"{CACHE}/gen_{TAG}"
    gen_path = lambda i, seed: f"{GEN_DIR}/f{i:05d}_s{seed}.png"
    has_person = [k for k in range(n) if joints[k] is not None]
    flicker = has_person[:3] if len(has_person) >= 3 else []         # 3 consecutive frames for the noise figure
    jobs = [(k, SEED) for k in has_person] + [(k, SEED + j) for j, k in enumerate(flicker) if j > 0]
    todo = [(k, s) for k, s in jobs if REGENERATE or not os.path.exists(gen_path(frame_idx[k], s))]

print("files")
print(f"  {VIDEO_PATH or f'{DATA_DIR}/{ACTION}.mov (or .mp4)':<36} {'found' if VIDEO_PATH else 'NOT FOUND'}")
print(f"  {DAVIS_PATH:<36} {'found' if os.path.exists(DAVIS_PATH) else 'NOT FOUND'}")
if not skip:
    print(f"  phone: {PHONE_WH[0]}x{PHONE_WH[1]}, {video.n_frames} frames ({video.n_frames / CAPTURE_FPS:.2f} s), loaded at {W}x{H}")
    print(f"  DAVIS: {davis.summary()}")
    print("values")
    print(f"  offset t_davis - t_phone  {sync.offset_s:.3f} s   {sync.source}" + ("" if sync.reliable or SYNC else "  (not reliable, see section 2)"))
    print(f"  motion start              phone {t_video0_s:.3f} s, DAVIS {t_davis0_s:.3f} s   {start_src}")
    print(f"  analysis window           DAVIS {t_davis0_s:.2f} to {t_davis0_s + frame_us[-1] / 1e6:.2f} s, {n} frame times "
          f"{1 / GEN_FPS:.2f} s apart   auto (CLIP_S = {CLIP_S} s)")
    print(f"  phone canvas, phone px    {box_phone}   {phone_src}")
    print(f"  DAVIS canvas, DAVIS px    {box_davis}   {davis_src}")
    print(f"  spatial mapping           {map_src}")
    print(f"  generated images          {len(jobs) - len(todo)} of {len(jobs)} cached in {GEN_DIR}")
print("sections")
gen_note = f"runs, {len(todo)} images to generate ({DEVICE})" if not skip and todo else "runs"
for name, ok in (("2 alignment", True), ("3 real events", True), ("4 simulated events", True), ("5 generated events", True),
                 ("6 side by side", True)):
    print(f"  {name:<20} " + (f"SKIPPED: {skip}" if skip else (gen_note if name.startswith("5") else "runs")))
if not skip:
    print(f"  {'6 overlay':<20} " + ("runs" if mapping is not None else "skipped: no spatial mapping (see above)"))
    print(f"  {'7 arms vs legs':<20} runs" + ("" if any(j is not None for j in aps_px) else ", real column skipped (no person on the APS frames)"))
print(f"checklist: {time.time() - t_sec:.1f} s")

## 1. Idea

Three ways to get event data of the same motion:

- **Real**: record it with the DAVIS346.
- **Simulated**: from high frame rate video with the event generation model of notebook 06. Physics: it models every pixel's log intensity, so it needs a high frame rate.
- **Generated**: from the 2D pose with the ControlEvents skeleton model (notebook 09). It needs only a skeleton and has learned from data what events of a moving person look like, but knows nothing about the actual brightness or timing.

All three are compared as event images. Real and simulated events are counted per pixel over `ACC_MS` around each frame time, without polarity (white events on black), which is the format of the skeleton model's output: it was trained on DHP-19 and CDEHP event images rendered without polarity.

## 2. Alignment

The two cameras start recording at different times. `sync_by_event_rate` (`evcam/align.py`) simulates events from the phone clip at reduced resolution and finds the offset $t_\text{davis} = t_\text{phone} + \Delta$ that maximizes the normalized cross-correlation of the two event rates (10 ms bins, each z-scored, so the different thresholds and resolutions do not matter), refined between bins with a parabola. It is reliable if the peak correlation is at least 0.5 and at least 1.2 times the highest other peak 0.5 s or more away. The first figure shows each event rate over its whole clip on its own clock, to read off a manual `SYNC`; the second shows both on the DAVIS clock after alignment. The motion start is the first clear rise of the real event rate after the phone clip starts; the compared frame times are `GEN_FPS` per second over `CLIP_S` from there.

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
else:
    plot_alignment(sync)
    print(describe(sync))
    print(f"motion start: phone {t_video0_s:.3f} s, DAVIS {t_davis0_s:.3f} s; {n} frame times over {frame_us[-1] / 1e6:.2f} s")
    if not sync.reliable and SYNC is None:
        print("NOTE: the automatic alignment is not reliable. If the curves in the second figure do not line up, pick one "
              "moment visible in both curves of the first figure and set SYNC = (t_phone_s, t_davis_s) in section 0. "
              "Continuing with the automatic value.")

## 3. Real events

DAVIS events counted per pixel over `ACC_MS` centred on each frame time, polarity ignored, mapped to the DAVIS canvas (checklist). Six of the frame times, evenly spaced, are shown here and in the later figures.

In [ ]:
t_sec = time.time()
if skip:
    print(f"SKIPPED: {skip}")
else:
    half = ACC_MS * 500
    real_img = [white_on_black(ce.to_canvas(count_image(in_window(real_ev, t - half, t + half), (Wd, Hd)), box_davis))
                for t in frame_us]
    ROWS = np.unique(np.linspace(0, n - 1, min(6, n)).round().astype(int))
    fig, ax = plt.subplots(1, len(ROWS), figsize=(2.8 * len(ROWS), 3), squeeze=False)
    for a, r in zip(ax[0], ROWS):
        a.imshow(real_img[r], cmap="gray", vmin=0, vmax=1); a.set_title(f"{frame_us[r] / 1e6:.2f} s", fontsize=10); a.axis("off")
    fig.suptitle(f"real events, {ACC_MS} ms, time since the motion start"); plt.tight_layout(); plt.show()
print(f"section 3: {time.time() - t_sec:.1f} s")

## 4. Simulated events (ESIM)

`simulate_events` (notebook 06) at $c$ = `C_SIM` on the phone frames of the analysed span plus `SIM_MARGIN_S` on each side, which keeps memory low; the margin before the motion start lets the reference levels settle on the still person. Accumulated as in section 3, in phone coordinates mapped to the phone canvas (checklist).

In [ ]:
t_sec = time.time()
if skip:
    print(f"SKIPPED: {skip}")
else:
    i0 = max(frame_idx[0] - int(SIM_MARGIN_S * CAPTURE_FPS), 0)
    i1 = min(frame_idx[-1] + int((SIM_MARGIN_S + ACC_MS / 2000) * CAPTURE_FPS) + 1, video.n_frames)
    sim = simulate_events(dataclasses.replace(video, frames=video.frames[i0:i1], frame_t=video.frame_t[i0:i1]), c=C_SIM)
    sim_ev = sim.events.copy(); sim_ev["t"] -= t_video0
    sim_img = [white_on_black(ce.to_canvas(count_image(in_window(sim_ev, t - half, t + half), (W, H)), box_work))
               for t in frame_us]
    print(f"{sim.n_events:,} events from frames {i0} to {i1 - 1}")
    fig, ax = plt.subplots(1, len(ROWS), figsize=(2.8 * len(ROWS), 3), squeeze=False)
    for a, r in zip(ax[0], ROWS):
        a.imshow(sim_img[r], cmap="gray", vmin=0, vmax=1); a.set_title(f"{frame_us[r] / 1e6:.2f} s", fontsize=10); a.axis("off")
    fig.suptitle(f"simulated events, c = {C_SIM}, {ACC_MS} ms"); plt.tight_layout(); plt.show()
print(f"section 4: {time.time() - t_sec:.1f} s")

## 5. Generated events (ControlEvents)

For each frame time: the OpenPose joints of the phone frame (checklist) → red skeleton on white, on the phone canvas → ControlEvents skeleton model (`combined_unet_all` + `combined_controlnet_all`), prompt "A moving human", as in notebook 09. Each frame is generated independently, so every frame starts from the same initial noise (a generator seeded with `SEED`): otherwise the appearance changes from frame to frame. Frames where OpenPose finds no person are skipped. Images are cached in a folder per crop, window and sampler settings, so changing any of them regenerates automatically; `REGENERATE` forces it.

In [ ]:
t_sec = time.time()
if skip:
    print(f"SKIPPED: {skip}")
else:
    os.makedirs(GEN_DIR, exist_ok=True)
    phone_rgb = {i: ce.to_canvas(phone_full[i], box_phone) for i in frame_idx}
    if todo:
        t_gen = time.time()
        pipe = ce.control_pipe(ce.SKELETON_UNET, ce.SKELETON_CONTROLNET, DEVICE, DTYPE)
        print(f"model loaded in {time.time() - t_gen:.0f} s")
        t_gen = time.time()
        for k, s in todo:
            ce.generate(pipe, ce.PROMPT_HUMAN, ce.draw_skeleton(joints[k]), seed=s, num_steps=NUM_STEPS,
                        guidance=GUIDANCE).save(gen_path(frame_idx[k], s))
        print(f"{len(todo)} images generated, {(time.time() - t_gen) / len(todo):.1f} s each ({DEVICE})")
        pipe = None; ce.free(DEVICE)
    gen_img = [None if joints[k] is None else
               np.asarray(Image.open(gen_path(frame_idx[k], SEED)).convert("L"), np.float32) / 255 for k in range(n)]
    print(f"{n} frames: {len(has_person)} with a person, {n - len(has_person)} skipped (no person), "
          f"{len(jobs) - len(todo)} of {len(jobs)} images from the cache")
print(f"section 5: {time.time() - t_sec:.1f} s")

**Fixed vs changing noise.** Three consecutive frames generated from the same initial noise (top) and with seed `SEED + k` for the k-th frame (bottom).

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
elif len(flicker) < 3:
    print("SKIPPED: fewer than 3 frames with a person")
else:
    fig, ax = plt.subplots(2, 3, figsize=(10, 6.8))
    for j, k in enumerate(flicker):
        for r, s in enumerate((SEED, SEED + j)):
            ax[r, j].imshow(Image.open(gen_path(frame_idx[k], s)))
            ax[r, j].set_title(f"{frame_us[k] / 1e6:.2f} s, seed {s}", fontsize=10); ax[r, j].axis("off")
    ax[0, 0].text(-0.08, 0.5, "same noise", transform=ax[0, 0].transAxes, rotation=90, va="center", ha="right")
    ax[1, 0].text(-0.08, 0.5, "seed per frame", transform=ax[1, 0].transAxes, rotation=90, va="center", ha="right")
    plt.tight_layout(); plt.show()

## 6. Side by side

Rows: six frame times. Columns: phone frame with the OpenPose skeleton, real events, simulated events, generated events. Each column is cropped around the person in its own camera, so the views differ slightly. The animation plays all frame times at `GEN_FPS` and is saved as `comparison.gif` in the cache folder.

In [ ]:
t_sec = time.time()
COLUMNS = ("phone + skeleton", "real (DAVIS)", f"simulated, c = {C_SIM}", "generated (ControlEvents)")


def tiles(k, size=SIZE):
    """The four images of frame k as (size, size, 3) uint8."""
    phone = phone_rgb[frame_idx[k]]
    phone = np.asarray(ce.draw_skeleton(joints[k], image=phone)) if joints[k] is not None else phone
    gray = lambda g: np.repeat((np.clip(g, 0, 1) * 255).astype(np.uint8)[..., None], 3, axis=2)
    gen = gray(gen_img[k]) if gen_img[k] is not None else np.zeros((SIZE, SIZE, 3), np.uint8)
    return [cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
            for im in (phone, gray(real_img[k]), gray(sim_img[k]), gen)]


if skip:
    print(f"SKIPPED: {skip}")
else:
    fig, ax = plt.subplots(len(ROWS), 4, figsize=(12, 3.1 * len(ROWS)), squeeze=False)
    for r, k in enumerate(ROWS):
        for c, im in enumerate(tiles(k)):
            ax[r, c].imshow(im); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
            if r == 0:
                ax[r, c].set_title(COLUMNS[c], fontsize=10)
        ax[r, 0].set_ylabel(f"{frame_us[k] / 1e6:.2f} s")
        if gen_img[k] is None:
            ax[r, 3].text(0.5, 0.5, "no person", color="w", ha="center", transform=ax[r, 3].transAxes)
    fig.suptitle(f"{ACTION}: time since the motion start, events over {ACC_MS} ms"); plt.tight_layout(); plt.show()

    T = 256
    header = np.full((28, 4 * T, 3), 255, np.uint8)
    for c, name in enumerate(COLUMNS):
        cv2.putText(header, name, (c * T + 6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    frames = []
    for k in range(n):
        row = np.concatenate(tiles(k, T), axis=1)
        cv2.putText(row, f"{frame_us[k] / 1e6:.2f} s", (6, T - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
        frames.append(Image.fromarray(np.concatenate([header, row], axis=0)))
    GIF = f"{CACHE}/comparison.gif"
    frames[0].save(GIF, save_all=True, append_images=frames[1:], duration=int(1000 / GEN_FPS), loop=0)
    print(f"saved {GIF}")
    display(ShowImage(filename=GIF))
print(f"section 6: {time.time() - t_sec:.1f} s")

**Real events on the generated image.** For three frame times, the real events mapped into the phone canvas with the spatial mapping of the checklist (DAVIS pixels → phone pixels → canvas), in red over the generated image. Where the generated events follow the real motion, the red falls on the bright parts.

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
elif mapping is None:
    print(f"SKIPPED: no spatial mapping between the cameras ({map_src.split(':')[0]}): keypoints and features both failed")
else:
    rows = [k for k in ROWS if gen_img[k] is not None][::2][:3]
    side = SIZE / (box_phone[2] - box_phone[0])
    to_canvas_m = np.array([[side, 0, -box_phone[0] * side], [0, side, -box_phone[1] * side], [0, 0, 1]])
    M = to_canvas_m @ np.linalg.inv(mapping)                                         # DAVIS px -> canvas px
    fig, ax = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4.2), squeeze=False)
    for a, k in zip(ax[0], rows):
        t = frame_us[k]
        real = white_on_black(count_image(in_window(real_ev, t - half, t + half), (Wd, Hd)))
        warped = cv2.warpPerspective(real, M, (SIZE, SIZE), flags=cv2.INTER_LINEAR)
        rgb = np.repeat(gen_img[k][..., None], 3, axis=2) * 0.7
        rgb[..., 0] = np.maximum(rgb[..., 0], warped); rgb[..., 1:] *= (1 - warped[..., None])
        a.imshow(np.clip(rgb, 0, 1)); a.set_title(f"{t / 1e6:.2f} s", fontsize=10); a.axis("off")
    fig.suptitle(f"generated (gray) with real events mapped onto it (red); mapping: {map_src}", fontsize=10)
    plt.tight_layout(); plt.show()

## 7. Where the events are: arms vs legs

For each method and frame time, the share of event pixels (value above `EVENT_THRESH` in the white-on-black image) within `LIMB_DIST` of the arm segments (shoulder-elbow-hand) and of the leg segments (hip-knee-foot), with the keypoints in each image's own coordinates: phone OpenPose keypoints for the simulated and generated images, OpenPose on the nearest DAVIS APS frame for the real events (checklist). For a wave the real events should concentrate on the arm, for a leg raise on the leg; this tests whether the generated events follow which limbs actually move or just the pose.

In [ ]:
t_sec = time.time()
EVENT_THRESH = 0.2         # event pixel: value above this in the [0, 1] image
LIMB_DIST = 0.04           # distance to a limb segment, fraction of the canvas side (about 20 px)


def limb_mask(xy, limbs):
    m = np.zeros((SIZE, SIZE), np.uint8)
    p = lambda j: tuple(int(v) for v in xy[j] * SIZE)
    for a, b in limbs:
        if not np.isnan(xy[[a, b]]).any():
            cv2.line(m, p(a), p(b), 1, 2 * int(LIMB_DIST * SIZE))
    return m.astype(bool)


def limb_shares(img, xy):
    """(arm share, leg share) of the event pixels of img; NaN without keypoints or events."""
    if img is None or xy is None:
        return np.nan, np.nan
    ev = img > EVENT_THRESH
    if not ev.any():
        return np.nan, np.nan
    return (ev & limb_mask(xy, ce.ARM_LIMBS)).sum() / ev.sum(), (ev & limb_mask(xy, ce.LEG_LIMBS)).sum() / ev.sum()


if skip:
    print(f"SKIPPED: {skip}")
else:
    if not any(j is not None for j in real_joints):
        print("SKIPPED (real column): " + ("no APS frames in the recording" if davis.frames is None else "OpenPose finds no person on the APS frames"))
    methods = {"real": (real_img, real_joints), "simulated": (sim_img, joints), "generated": (gen_img, joints)}
    shares = {m: np.array([limb_shares(imgs[k], kps[k]) for k in range(n)]) for m, (imgs, kps) in methods.items()}
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
    for m, sh in shares.items():
        for a, col in zip(ax, (0, 1)):
            a.plot(frame_us / 1e6, sh[:, col], "o-", ms=3, lw=1, label=m)
    for a, name in zip(ax, ("arms", "legs")):
        a.set_xlabel("time since the motion start (s)"); a.set_title(f"share of event pixels near the {name}")
        a.set_ylabel("share of event pixels (0 to 1)")
    ax[0].legend(); plt.tight_layout(); plt.show()
    print(f"{'mean share':<12}{'arms':>8}{'legs':>8}")
    for m, sh in shares.items():
        print(f"{m:<12}{np.nanmean(sh[:, 0]) if np.isfinite(sh[:, 0]).any() else np.nan:>8.2f}"
              f"{np.nanmean(sh[:, 1]) if np.isfinite(sh[:, 1]).any() else np.nan:>8.2f}")
print(f"section 7: {time.time() - t_sec:.1f} s")

In [ ]:
print(f"notebook runtime {time.time() - T_NOTEBOOK:.0f} s")

## References

1. H. Rebecq, D. Gehrig, D. Scaramuzza, "ESIM: an Open Event Camera Simulator", Conference on Robot Learning (CoRL) 2018.
2. Y. Hu, Y. Xue, S. Klenk, D. Cremers, G. Pons-Moll, "ControlEvents: Controllable Synthesis of Event Camera Data with Foundational Prior from Image Diffusion Models", WACV 2026, [arXiv:2509.22864](https://arxiv.org/abs/2509.22864). Weights: [huggingface.co/EventDiffusion](https://huggingface.co/EventDiffusion) (MIT).
3. R. Rombach, A. Blattmann, D. Lorenz, P. Esser, B. Ommer, "High-Resolution Image Synthesis with Latent Diffusion Models", CVPR 2022 (Stable Diffusion; weights under CreativeML OpenRAIL-M).
4. L. Zhang, A. Rao, M. Agrawala, "Adding Conditional Control to Text-to-Image Diffusion Models", ICCV 2023 (ControlNet).
5. W. Zhao, L. Bai, Y. Rao, J. Zhou, J. Lu, "UniPC: A Unified Predictor-Corrector Framework for Fast Sampling of Diffusion Models", NeurIPS 2023.
6. Z. Cao, G. Hidalgo, T. Simon, S.-E. Wei, Y. Sheikh, "OpenPose: Realtime Multi-Person 2D Pose Estimation Using Part Affinity Fields", IEEE TPAMI 43(1), 2021 (online 2019).
7. E. Calabrese, G. Taverni, C. Awai Easthope, S. Skriabine, F. Corradi, L. Longinotti, K. Eng, T. Delbruck, "DHP19: Dynamic Vision Sensor 3D Human Pose Dataset", CVPR Workshops 2019.
8. Z. Shao, X. Wang, W. Zhou, W. Wang, J. Yang, Y. Li, "A Temporal Densely Connected Recurrent Network for Event-based Human Pose Estimation", Pattern Recognition 147, 2024 (CDEHP).
9. S. Zou, C. Guo, X. Zuo, S. Wang, P. Wang, X. Hu, S. Chen, M. Gong, L. Cheng, "EventHPE: Event-based 3D Human Pose and Shape Estimation", ICCV 2021 (Event-HPE, 15 ms event windows).
10. C. Brandli, R. Berner, M. Yang, S.-C. Liu, T. Delbruck, "A 240×180 130 dB 3 µs Latency Global Shutter Spatiotemporal Vision Sensor", IEEE Journal of Solid-State Circuits 49(10), 2014 (DAVIS sensor).